# A1.2 — CNN vs Transformer trên Food-101 (ResNet-50 vs DeiT-S)

Notebook chạy toàn bộ thí nghiệm theo protocol A1.1 (`reports/A1.1_Dataset-Protocol.pdf`).

- **Runtime:** Colab **A100** (protocol yêu cầu A100 cho run chính thức; L4/T4 chỉ để debug).
- Code ở `A1/src/`, kết quả ghi vào `A1/runs/<RUN>_s<seed>/` trên Drive → mất kết nối thì chạy lại cell, run đã xong sẽ được bỏ qua, run dở sẽ **resume** từ `last.pt`.

| ID | Mô hình | Chế độ | Epoch | Seed |
|---|---|---|---|---|
| R1 / R5 | ResNet-50 / DeiT-S | From-scratch | 100 | 42 |
| R2 / R6 | ResNet-50 / DeiT-S | Freeze toàn bộ (feature cache) | 30 | 42, 43, 44 |
| R3 / R7 | ResNet-50 / DeiT-S | Freeze một phần | 15 | 42, 43, 44 |
| R4 / R8 | ResNet-50 / DeiT-S | Full finetune | 15 | 42, 43, 44 |

## 1. Drive, GPU, thư viện

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Thac si/Hoc ki 3/Hoc sau va ung dung/Exercises/SDH_HK261_CO5085/A1')
assert (ROOT/'src/train.py').is_file(), ROOT
os.chdir(ROOT)
print('Python:', sys.version)
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout)
result.check_returncode()
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## 2. Dữ liệu Food-101 → ổ cục bộ `/content/data`

Lần đầu: tải từ ETH (~5 GB) và lưu bản `.tar.gz` lên Drive. Các phiên sau: copy từ Drive rồi giải nén (nhanh hơn, không phụ thuộc server ETH). Không đọc ảnh trực tiếp từ Drive.

In [ ]:
import os, shutil, time
DATA_ROOT = '/content/data'
DRIVE_TAR = '/content/drive/MyDrive/datasets/food-101.tar.gz'
os.makedirs(DATA_ROOT, exist_ok=True)
t0 = time.time()
if not os.path.isdir(f'{DATA_ROOT}/food-101/images'):
    if os.path.exists(DRIVE_TAR):
        shutil.copy(DRIVE_TAR, f'{DATA_ROOT}/food-101.tar.gz')
        subprocess.run(['tar', '-xzf', '/content/data/food-101.tar.gz', '-C', '/content/data'], check=True)
    else:
        from torchvision.datasets import Food101
        Food101(DATA_ROOT, split='train', download=True)
        os.makedirs(os.path.dirname(DRIVE_TAR), exist_ok=True)
        shutil.copy(f'{DATA_ROOT}/food-101.tar.gz', DRIVE_TAR)
print(f'Sẵn sàng sau {time.time() - t0:.0f}s')
subprocess.run(['ls', '/content/data/food-101'], check=True)


## 3. Split train/val (stratified 10%, seed 42) — chỉ tạo một lần, commit `A1/splits/`

In [ ]:
from src.data import make_splits
if not os.path.exists('splits/train_idx.json'):
    make_splits(DATA_ROOT, 'splits', val_frac=0.1, seed=42)
else:
    print('Đã có splits/, dùng lại để mọi run chung một split')


## 4. EDA

In [ ]:
import json, random
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
from torchvision.datasets import Food101
os.makedirs('figures', exist_ok=True)
base = Food101(DATA_ROOT, split='train')
classes = base.classes
print('Số lớp:', len(classes), '| ảnh train chính thức:', len(base), '| ảnh/lớp:', np.bincount(base._labels).min(), '-', np.bincount(base._labels).max())

# Phân bố kích thước ảnh (mẫu 2000 ảnh)
random.seed(0)
sizes = np.array([Image.open(base._image_files[i]).size for i in random.sample(range(len(base)), 2000)])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(sizes[:, 0], bins=40); ax[0].set_title('Chiều rộng (px)')
ax[1].hist(sizes[:, 1], bins=40); ax[1].set_title('Chiều cao (px)')
fig.tight_layout(); fig.savefig('figures/eda_sizes.png', dpi=130); plt.show()

# Lưới ảnh mẫu: 24 lớp ngẫu nhiên, mỗi lớp 1 ảnh
labels = np.array(base._labels)
pick = random.sample(range(len(classes)), 24)
fig, axes = plt.subplots(4, 6, figsize=(15, 10))
for ax, c in zip(axes.flat, pick):
    i = int(np.where(labels == c)[0][0])
    ax.imshow(Image.open(base._image_files[i]).convert('RGB')); ax.set_title(classes[c], fontsize=8); ax.axis('off')
fig.tight_layout(); fig.savefig('figures/eda_samples.png', dpi=110); plt.show()


## 5. Smoke test (vài batch, kiểm tra pipeline trước khi tốn compute unit)

In [ ]:
subprocess.run([sys.executable, '-m', 'src.train', '--run', 'R1', 'R2', 'R3', 'R8', '--seeds', '42', '--smoke', '--data_root', '/content/data', '--out_root', 'runs'], check=True)


## 6. (Tùy chọn) Dò LR backbone cho full finetune: {5e-5, 1e-4, 3e-4}

Protocol mục 7. Tốn thêm 6 run × 15 epoch. Nếu thiếu thời gian thì bỏ qua và ghi rõ trong báo cáo là dùng LR mặc định 1e-4.

In [ ]:
# for lr in ['5e-5', '3e-4']:   # 1e-4 chính là run R4/R8 seed 42
#     !python -m src.train --run R4 R8 --seeds 42 --lr_backbone {lr} --data_root /content/data --out_root runs


## 7. Chạy các run chính

Thứ tự ưu tiên: run rẻ trước (freeze toàn bộ, finetune), from-scratch 100 epoch để sau cùng. Mỗi lệnh tự bỏ qua run đã có `metrics.json` và resume run dở dang.

In [ ]:
# 7a. Freeze toàn bộ backbone (trích feature 1 lần cho mỗi mô hình, 3 seed train head)
subprocess.run([sys.executable, '-m', 'src.train', '--run', 'R2', 'R6', '--data_root', '/content/data', '--out_root', 'runs'], check=True)


In [ ]:
# 7b. Full finetune + freeze một phần, seed 42 trước
subprocess.run([sys.executable, '-m', 'src.train', '--run', 'R4', 'R8', 'R3', 'R7', '--seeds', '42', '--data_root', '/content/data', '--out_root', 'runs'], check=True)


In [ ]:
# 7c. From-scratch 100 epoch (tốn nhất)
subprocess.run([sys.executable, '-m', 'src.train', '--run', 'R1', 'R5', '--data_root', '/content/data', '--out_root', 'runs'], check=True)


In [ ]:
# 7d. Seed 43, 44 cho các run pretrained (mean ± std)
subprocess.run([sys.executable, '-m', 'src.train', '--run', 'R4', 'R8', 'R3', 'R7', '--seeds', '43', '44', '--data_root', '/content/data', '--out_root', 'runs'], check=True)


In [ ]:
# 7e. (Mở rộng, tùy chọn) ConvNeXt-T vs Swin-T
# !python -m src.train --run X2 X6 X4 X8 X3 X7 X1 X5 --data_root /content/data --out_root runs


## 8. Tổng hợp bảng + biểu đồ

In [ ]:
subprocess.run([sys.executable, '-m', 'src.summarize', '--out_root', 'runs', '--fig_dir', 'figures'], check=True)
from IPython.display import Image as IImage, display, Markdown
display(Markdown(open('summary.md', encoding='utf-8').read()))
for f in ['curves', 'tradeoff', 'pretrain_gap', 'confusion_resnet50', 'confusion_deit_s']:
    if os.path.exists(f'figures/{f}.png'):
        display(IImage(f'figures/{f}.png'))


In [ ]:
# Lưới ảnh đoán sai của run tốt nhất mỗi mô hình (phân tích lỗi: món giống nhau / nhiều món / nhãn sai)
import pandas as pd
from src.summarize import misclassified_grid
classes = json.load(open('splits/classes.json'))
df = pd.read_csv('runs_all.csv')
for model, g in df.groupby('model'):
    best = g.sort_values('best_val_top1', ascending=False).iloc[0]
    out = f'figures/errors_{model}.png'
    misclassified_grid(best['dir'], DATA_ROOT, classes, out)
    print(model, best['run_id'], 'seed', best['seed']); display(IImage(out))
